# Load pretrain weights from openAI

In [2]:
# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt).
# Source for "Build a Large Language Model From Scratch"
#   - https://www.manning.com/books/build-a-large-language-model-from-scratch
# Code: https://github.com/rasbt/LLMs-from-scratch


import os

import requests
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm


def download_and_load_gpt2(model_size, models_dir):
    # Validate model size
    allowed_sizes = ("124M", "355M", "774M", "1558M")
    if model_size not in allowed_sizes:
        raise ValueError(f"Model size not in {allowed_sizes}")

    # Define paths
    model_dir = os.path.join(models_dir, model_size)
    base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
    backup_base_url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2"
    filenames = [
        "checkpoint", "encoder.json", "hparams.json",
        "model.ckpt.data-00000-of-00001", "model.ckpt.index",
        "model.ckpt.meta", "vocab.bpe"
    ]

    # Download files
    os.makedirs(model_dir, exist_ok=True)
    for filename in filenames:
        file_url = os.path.join(base_url, model_size, filename)
        backup_url = os.path.join(backup_base_url, model_size, filename)
        file_path = os.path.join(model_dir, filename)
        download_file(file_url, file_path, backup_url)

    # Load settings and params
    tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
    settings = json.load(open(os.path.join(model_dir, "hparams.json"), "r", encoding="utf-8"))
    params = load_gpt2_params_from_tf_ckpt(tf_ckpt_path, settings)

    return settings, params


def download_file(url, destination, backup_url=None):
    def _attempt_download(download_url):
        response = requests.get(download_url, stream=True, timeout=60)
        response.raise_for_status()

        file_size = int(response.headers.get("Content-Length", 0))

        # Check if file exists and has same size
        if os.path.exists(destination):
            file_size_local = os.path.getsize(destination)
            if file_size and file_size == file_size_local:
                print(f"File already exists and is up-to-date: {destination}")
                return True

        block_size = 1024  # 1 KB
        desc = os.path.basename(download_url)
        with tqdm(total=file_size, unit="iB", unit_scale=True, desc=desc) as progress_bar:
            with open(destination, "wb") as file:
                for chunk in response.iter_content(chunk_size=block_size):
                    if chunk:
                        file.write(chunk)
                        progress_bar.update(len(chunk))
        return True

    try:
        if _attempt_download(url):
            return
    except requests.exceptions.RequestException:
        if backup_url is not None:
            print(f"Primary URL ({url}) failed. Attempting backup URL: {backup_url}")
            try:
                if _attempt_download(backup_url):
                    return
            except requests.exceptions.RequestException:
                pass

        error_message = (
            f"Failed to download from both primary URL ({url})"
            f"{' and backup URL (' + backup_url + ')' if backup_url else ''}."
            "\nCheck your internet connection or the file availability.\n"
            "For help, visit: https://github.com/rasbt/LLMs-from-scratch/discussions/273"
        )
        print(error_message)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


# Alternative way using `requests`
"""
def download_file(url, destination):
    # Send a GET request to download the file in streaming mode
    response = requests.get(url, stream=True)

    # Get the total file size from headers, defaulting to 0 if not present
    file_size = int(response.headers.get("content-length", 0))

    # Check if file exists and has the same size
    if os.path.exists(destination):
        file_size_local = os.path.getsize(destination)
        if file_size == file_size_local:
            print(f"File already exists and is up-to-date: {destination}")
            return

    # Define the block size for reading the file
    block_size = 1024  # 1 Kilobyte

    # Initialize the progress bar with total file size
    progress_bar_description = url.split("/")[-1]  # Extract filename from URL
    with tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
        # Open the destination file in binary write mode
        with open(destination, "wb") as file:
            # Iterate over the file data in chunks
            for chunk in response.iter_content(block_size):
                progress_bar.update(len(chunk))  # Update progress bar
                file.write(chunk)  # Write the chunk to the file
"""


def load_gpt2_params_from_tf_ckpt(ckpt_path, settings):
    # Initialize parameters dictionary with empty blocks for each layer
    params = {"blocks": [{} for _ in range(settings["n_layer"])]}

    # Iterate over each variable in the checkpoint
    for name, _ in tf.train.list_variables(ckpt_path):
        # Load the variable and remove singleton dimensions
        variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))

        # Process the variable name to extract relevant parts
        variable_name_parts = name.split("/")[1:]  # Skip the 'model/' prefix

        # Identify the target dictionary for the variable
        target_dict = params
        if variable_name_parts[0].startswith("h"):
            layer_number = int(variable_name_parts[0][1:])
            target_dict = params["blocks"][layer_number]

        # Recursively access or create nested dictionaries
        for key in variable_name_parts[1:-1]:
            target_dict = target_dict.setdefault(key, {})

        # Assign the variable array to the last key
        last_key = variable_name_parts[-1]
        target_dict[last_key] = variable_array

    return params

In [3]:
setting, params = download_and_load_gpt2(model_size="124M", models_dir='gpt2')

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [13]:
print("setting: ", setting)
print("params: ", len(params['blocks']))

setting:  {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
params:  12


In [ ]:
model_config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embedding_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout": 0.1,
    "qkv_bias": True, # OPENAI CONFIG
}

In [36]:
# place holder for GPT2 architecture implementation
import torch
import torch.nn as nn

    
class DummyLayerNorm(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.epsilon = 1e-5
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.epsilon)
        x = self.scale * normalized + self.shift
        return x
    
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config['embedding_dim'], 4 * config['embedding_dim']),
            nn.GELU(),
            nn.Linear(4 * config['embedding_dim'], config['embedding_dim'])
        )
    def forward(self, x):
        return self.layers(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_lengh, dropout, qkv_bias=False):
        super().__init__()
        print("initializing multi head attention with d_in:", d_in, "d_out:", d_out, "num_heads:", num_heads, "context_length:", context_lengh, "dropout:", dropout)
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_size = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Dropout = nn.Dropout(dropout)
        self.OutProj = nn.Linear(d_out, d_out) # output projection to combine heads
        self.register_buffer('mask', torch.tril(torch.ones(context_lengh, context_lengh), diagonal=0)) # precompute mask for max context length of 1000

    def forward(self,x):
        batch_size, num_tokens, d_in = x.shape
        querys = self.W_query(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        keys = self.W_key(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        values = self.W_value(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        # print("querys shape:", querys.shape)
        # print("keys shape:", keys.shape)
        # print("values shape:", values.shape)
       
        querys = querys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)

        attn_scores = querys @ keys.transpose(-2,-1) # (batch_size, num_heads, num_tokens, head_size) @ (batch_size, num_heads, head_size, num_tokens) --> (batch_size, num_heads, num_tokens, num_tokens)
        # print("before masked: ", attn_scores)
        masked_attn_scores = attn_scores.masked_fill(self.mask[:num_tokens, :num_tokens] == 0, float('-inf')) # (batch_size, num_heads, num_tokens, num_tokens)
        # print("mask: ", self.mask)
        # print("after masked: ", masked_attn_scores)
        attn_weights = torch.softmax(masked_attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # (batch_size, num_heads, num_tokens, num_tokens)
        attn_weights = self.Dropout(attn_weights) # (batch_size, num_heads, num_tokens, num_tokens)
        context_vector = attn_weights @ values # (batch_size, num_heads, num_tokens, num_tokens) @ (batch_size, num_heads, num_tokens, head_size) --> (batch_size, num_heads, num_tokens, head_size)

        context_vector = context_vector.transpose(1,2) # (batch_size, num_tokens, num_heads, head_size)
        # print("context: ", context_vector)
        #combine heads
        context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.num_heads * self.head_size) # (batch_size, num_tokens, d_out)
        # print("combined context: ", context_vector)

        context_vector = self.OutProj(context_vector) # (batch_size, num_tokens, d_out)
        return context_vector

    
class DummyTransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(d_in=config['embedding_dim'],
            d_out=config['embedding_dim'],
            context_lengh=config['context_length'],
            num_heads=config['num_heads'],
            dropout=config['dropout'],
            qkv_bias=config['qkv_bias'])
        self.feed_forward = FeedForward(config)
        self.norm1 = DummyLayerNorm(config['embedding_dim'])
        self.norm2 = DummyLayerNorm(config['embedding_dim'])
        self.dropout = nn.Dropout(config['dropout'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut
        return x

class DummyGPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config['vocab_size'], config['embedding_dim'])
        self.position_embedding = nn.Embedding(config['context_length'], config['embedding_dim'])
        self.dropout_embedding = nn.Dropout(config['dropout'])
        self.transformer_blocks = nn.Sequential(*[DummyTransformerBlock(config) for _ in range(config['num_layers'])])
        self.final_norm = DummyLayerNorm(config['embedding_dim'])
        self.output_head = nn.Linear(config['embedding_dim'], config['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_length = in_idx.shape
        token_embeddings = self.token_embedding(in_idx)
        position_embeddings = self.position_embedding(torch.arange(seq_length))
        x = self.dropout_embedding(token_embeddings + position_embeddings)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits



In [37]:
gpt = DummyGPT2(model_config)
gpt.eval()

initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head atte

DummyGPT2(
  (token_embedding): Embedding(50257, 768)
  (position_embedding): Embedding(1024, 768)
  (dropout_embedding): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): DummyTransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (Dropout): Dropout(p=0.1, inplace=False)
        (OutProj): Linear(in_features=768, out_features=768, bias=True)
      )
      (feed_forward): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): DummyLayerNorm()
      (norm2): DummyLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): DummyTransf

In [38]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [39]:
import numpy as np
def load_weights_into_gpt(gpt, params):
    gpt.position_embedding.weight = assign(gpt.position_embedding.weight, params['wpe'])
    gpt.token_embedding.weight = assign(gpt.token_embedding.weight, params['wte'])
    
    for b in range(len(params['blocks'])):
        # weight self-attention block
        q_w, k_w, v_w = np.split((params['blocks'][b]['attn']['c_attn'])['w'], 3, axis=-1)
        gpt.transformer_blocks[b].attention.W_query.weight = assign(gpt.transformer_blocks[b].attention.W_query.weight, q_w.T)
        gpt.transformer_blocks[b].attention.W_key.weight = assign(gpt.transformer_blocks[b].attention.W_key.weight, k_w.T)
        gpt.transformer_blocks[b].attention.W_value.weight = assign(gpt.transformer_blocks[b].attention.W_value.weight, v_w.T)

        # bias self-attention block
        b_w, b_k, b_v = np.split((params['blocks'][b]['attn']['c_attn'])['b'], 3, axis=-1)
        gpt.transformer_blocks[b].attention.W_query.bias = assign(gpt.transformer_blocks[b].attention.W_query.bias, b_w)
        gpt.transformer_blocks[b].attention.W_key.bias = assign(gpt.transformer_blocks[b].attention.W_key.bias, b_k)
        gpt.transformer_blocks[b].attention.W_value.bias = assign(gpt.transformer_blocks[b].attention.W_value.bias, b_v)

        # OutProj self-attention block
        gpt.transformer_blocks[b].attention.OutProj.weight = assign(gpt.transformer_blocks[b].attention.OutProj.weight,
                                                                    params['blocks'][b]['attn']['c_proj']['w'].T)
        gpt.transformer_blocks[b].attention.OutProj.bias = assign(gpt.transformer_blocks[b].attention.OutProj.bias,
                                                                    params['blocks'][b]['attn']['c_proj']['b'].T)
         
        # feed forward layers
        gpt.transformer_blocks[b].feed_forward.layers[0].weight = assign(gpt.transformer_blocks[b].feed_forward.layers[0].weight,
                                                                 params['blocks'][b]['mlp']['c_fc']['w'].T)
        gpt.transformer_blocks[b].feed_forward.layers[0].bias = assign(gpt.transformer_blocks[b].feed_forward.layers[0].bias,
                                                                 params['blocks'][b]['mlp']['c_fc']['b'])
        
        gpt.transformer_blocks[b].feed_forward.layers[2].weight = assign(gpt.transformer_blocks[b].feed_forward.layers[2].weight,
                                                                 params['blocks'][b]['mlp']['c_proj']['w'].T)
        gpt.transformer_blocks[b].feed_forward.layers[2].bias = assign(gpt.transformer_blocks[b].feed_forward.layers[2].bias,
                                                                 params['blocks'][b]['mlp']['c_proj']['b'])
        
        gpt.transformer_blocks[b].norm1.scale = assign(gpt.transformer_blocks[b].norm1.scale,
                                                       params['blocks'][b]['ln_1']['g'])
        gpt.transformer_blocks[b].norm1.shift = assign(gpt.transformer_blocks[b].norm1.scale,
                                                       params['blocks'][b]['ln_1']['b'])
        
        gpt.transformer_blocks[b].norm2.scale = assign(gpt.transformer_blocks[b].norm2.scale,
                                                       params['blocks'][b]['ln_2']['g'])
        gpt.transformer_blocks[b].norm2.shift = assign(gpt.transformer_blocks[b].norm2.scale,
                                                       params['blocks'][b]['ln_2']['b'])
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params['g'])
    gpt.final_norm.shift = assign(gpt.final_norm.scale, params['b'])
    gpt.output_head.weight = assign(gpt.output_head.weight, params['wte'])

load_weights_into_gpt(gpt, params)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpt.to(device)

DummyGPT2(
  (token_embedding): Embedding(50257, 768)
  (position_embedding): Embedding(1024, 768)
  (dropout_embedding): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): DummyTransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (Dropout): Dropout(p=0.1, inplace=False)
        (OutProj): Linear(in_features=768, out_features=768, bias=True)
      )
      (feed_forward): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): DummyLayerNorm()
      (norm2): DummyLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): DummyTransf

In [43]:
import tiktoken

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):

    for _ in range(max_new_tokens):                      #1
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        if top_k is not None:                            #2
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]

            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )

        if temperature > 0.0:                           #3
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )
        else:                                            #4
            idx_next = torch.argmax(
                logits,
                dim=-1,
                keepdim=True
            )

        if idx_next == eos_id:                          #5
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)    #1
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)                            #2
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids(start_context, tokenizer).to(device),
    max_new_tokens=1000,
    context_size=model_config["context_length"],
    top_k=50,
    temperature=0.5,
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you forward, but it's not easy.

What is the biggest challenge?

The biggest challenge is that you have to learn to make the most of your life, and to do that, you have to be able to keep going. You have to learn to be able to do everything you want to do. You have to learn how to be a good person. You have to learn to be able to do all the things that you want to do.

This is what I mean by "learning to be a good person."

You have to learn how to be a good person.

You have to learn to be able to do everything you want to do.

You have to learn how to be a good person.

You have to learn to be able to do all the things that you want to do.

This is what I mean by "learning to be a good person."

You have to learn to be able to do everything you want to do.

You have to learn to be able to do all the things that you want to do.

This is what I mean by "learning to be a good person."

You have to learn to be able to do all the things that you want to do